# Anomaly Generation

In [1]:
import numpy as np
import pickle
import random


In [8]:

# --- CONFIG ---
INPUT_PATH = 'data/final_models_input/train_dataset.pkl'
OUTPUT_PATH = 'data/final_models_input/train_dataset_with_anomalies.pkl'

# Indices
IDX_POWER = 0
IDX_GV = 1
IDX_P_DOWN = 2
IDX_P_UP = 3

# --- PREVIOUS ANOMALIES (Keep these) ---
def anomaly_seal_leakage(signal, severity=15.0):
    T = len(signal)
    start_idx = int(T * 0.6) # Tail
    ramp = np.linspace(0, severity, T - start_idx)
    noise = np.random.normal(0, 0.5, size=len(ramp))
    anom = signal.copy()
    anom[start_idx:] += (ramp + noise)
    return anom

def anomaly_delayed_closure(signal, delay_steps=30):
    anom = np.roll(signal, delay_steps)
    anom[:delay_steps] = signal[0]
    return anom

def anomaly_sensor_offset(signal, magnitude=-15.0):
    return signal + magnitude

def anomaly_sensor_drift(signal, magnitude=30.0):
    drift = np.linspace(0, magnitude, len(signal))
    return signal + drift

# --- NEW ANOMALIES (Closing Transient Specific) ---

def anomaly_water_hammer_spike(signal, intensity=1.5):
    """
    Simulates a dangerous Water Hammer (Pressure Spike).
    Logic: Finds the max peak of pressure and amplifies it.
    """
    anom = signal.copy()
    
    # Find the region where pressure is highest (The spike)
    # usually in the first 20-40% of the window for upstream, 
    # or right at the closure moment for downstream.
    
    # Let's target the peak
    peak_idx = np.argmax(anom)
    
    # Define a window around the peak to amplify
    window = 20 # +/- 20 steps
    start = max(0, peak_idx - window)
    end = min(len(anom), peak_idx + window)
    
    # Amplify the spike (e.g., multiply by 1.5)
    # We add an offset proportional to the value to preserve shape
    spike_shape = anom[start:end]
    amplification = spike_shape * (intensity - 1.0) 
    
    anom[start:end] += amplification
    return anom

# def anomaly_jerky_movement(signal, freq=0.5, amp=0.05):
#     """
#     Simulates Stick-Slip or Hydraulic instability in Guide Vanes.
#     Adds a sinusoidal wobble to the smooth closing curve.
#     """
#     T = len(signal)
#     t = np.arange(T)
    
#     # Create wobble (Sine wave)
#     # freq controls how fast it wobbles
#     # amp controls how strong the wobble is (relative to signal magnitude)
#     wobble = np.sin(2 * np.pi * freq * t) * amp
    
#     # Add wobble only during the transition (where signal is changing)
#     # We estimate transition roughly by checking where derivative is non-zero
#     # Or just apply to the whole curve for simplicity
    
#     return signal + wobble
def anomaly_stick_slip_staircase(signal, step_size=20):
    """
    Simulates severe friction (Stick-Slip).
    Instead of a smooth curve, the valve moves in 'steps'.
    Physical meaning: Servo pushes -> Valve stuck -> Valve jumps -> Valve stuck.
    """
    anom = signal.copy()
    T = len(signal)
    
    # We iterate through the signal and hold values constant for 'step_size'
    for i in range(0, T, step_size):
        # Define the window
        end = min(i + step_size, T)
        
        # Force the signal to stay at the value of the start of the step
        # This creates a flat plateau
        value_at_start = anom[i]
        anom[i:end] = value_at_start
        
        # The 'jump' happens automatically at 'end' when the next loop starts
        # and picks the REAL value from the original signal, creating a sharp drop.
        
    return anom

def anomaly_pid_hunting(signal, intensity=0.1):
    """
    Simulates Controller Instability at the end of closure.
    The valve oscillates (bounces) before settling at 0.
    """
    anom = signal.copy()
    T = len(signal)
    
    # Only apply to the last 20% (Settling phase)
    start_idx = int(T * 0.8)
    
    # Create a decaying sine wave
    t = np.arange(T - start_idx)
    # Frequency: Fast oscillation
    # Decay: Exp(-t) so it settles down
    oscillation = np.sin(t * 0.5) * np.exp(-t * 0.05) * intensity
    
    # Add to signal
    anom[start_idx:] += oscillation
    
    # Clip to physical limits (e.g. can't go below -0.05) if necessary
    # anom = np.clip(anom, -0.1, 1.1) 
    
    return anom

def anomaly_signal_dropout(signal, num_drops=3):
    """
    Simulates loose wiring / sensor failure.
    Randomly drops values to 0 for 1-2 time steps.
    """
    anom = signal.copy()
    T = len(signal)
    
    for _ in range(num_drops):
        idx = np.random.randint(5, T-5)
        # Drop to 0 (or a very low value relative to range)
        anom[idx] = 0.0
        anom[idx+1] = 0.0 # 2-step dropout
        
    return anom


In [9]:

# --- GENERATION LOOP ---

def generate_comprehensive_anomalies():
    print(f"Loading Raw Train Data from {INPUT_PATH}...")
    with open(INPUT_PATH, 'rb') as f:
        data = pickle.load(f)
    X_raw = data['X'][-500:] 
    
    X_anom = []
    y_labels = [] 
    descriptions = []
    
    print("Generating comprehensive anomaly set...")
    
    for sample in X_raw:
        # 0. Normal
        X_anom.append(sample)
        y_labels.append(0)
        descriptions.append("Normal")
        
        # 1. Seal Leakage (Pressure Down)
        s1 = sample.copy()
        s1[:, IDX_P_DOWN] = anomaly_seal_leakage(s1[:, IDX_P_DOWN])
        X_anom.append(s1)
        y_labels.append(1)
        descriptions.append("Seal Leakage")
        
        # 2. Delayed Closure (All Signals, primarily P_Down)
        s2 = sample.copy()
        s2[:, IDX_P_DOWN] = anomaly_delayed_closure(s2[:, IDX_P_DOWN])
        X_anom.append(s2)
        y_labels.append(2)
        descriptions.append("Delayed Closure")
        
        # 3. Sensor Offset (Pressure Up)
        s3 = sample.copy()
        s3[:, IDX_P_UP] = anomaly_sensor_offset(s3[:, IDX_P_UP])
        X_anom.append(s3)
        y_labels.append(3)
        descriptions.append("Sensor Offset")
        
        # 4. Sensor Drift (Power)
        s4 = sample.copy()
        s4[:, IDX_POWER] = anomaly_sensor_drift(s4[:, IDX_POWER])
        X_anom.append(s4)
        y_labels.append(4)
        descriptions.append("Sensor Drift")
        
        # --- NEW ANOMALIES ---
        
        # 5. Water Hammer Spike (Pressure Up) - Critical Safety
        s5 = sample.copy()
        # Increase spike by 30% (dangerous pressure)
        s5[:, IDX_P_UP] = anomaly_water_hammer_spike(s5[:, IDX_P_UP], intensity=1.3) 
        X_anom.append(s5)
        y_labels.append(5)
        descriptions.append("Water Hammer Spike")
        
        # 6. Stick-Slip (Staircase) - Replaces "Jerky"
        # This creates a distinctive "Lego block" look that is definitely anomalous
        s6 = sample.copy()
        s6[:, IDX_GV] = anomaly_stick_slip_staircase(s6[:, IDX_GV], step_size=25)
        X_anom.append(s6)
        y_labels.append(6)
        descriptions.append("Stick-Slip (Staircase)")

        # 7. PID Hunting (Overshoot) - New Type
        s7 = sample.copy()
        s7[:, IDX_GV] = anomaly_pid_hunting(s7[:, IDX_GV], intensity=0.15)
        X_anom.append(s7)
        y_labels.append(7)
        descriptions.append("PID Hunting (Oscillation)")
        
        # 8. Signal Dropout (Pressure Down) - Sensor Fault
        s8 = sample.copy()
        s8[:, IDX_P_DOWN] = anomaly_signal_dropout(s8[:, IDX_P_DOWN])
        X_anom.append(s8)
        y_labels.append(8)
        descriptions.append("Signal Dropout")

    # Save
    save_dict = {"X": np.array(X_anom), "y": np.array(y_labels), "descriptions": descriptions}
    with open(OUTPUT_PATH, 'wb') as f:
        pickle.dump(save_dict, f)
        
    print(f"Saved {len(X_anom)} samples to {OUTPUT_PATH}")


In [10]:

if __name__ == "__main__":
    generate_comprehensive_anomalies()

Loading Raw Train Data from data/final_models_input/train_dataset.pkl...
Generating comprehensive anomaly set...
Saved 4500 samples to data/final_models_input/train_dataset_with_anomalies.pkl


## Visualize the anomalies

In [11]:
import matplotlib.pyplot as plt
import numpy as np
import pickle
import ipywidgets as widgets
from ipywidgets import interact, Layout


In [12]:

# --- CONFIGURATION ---
# Make sure this matches the OUTPUT_PATH from Cell 1
DATA_PATH = 'data/final_models_input/train_dataset_with_anomalies.pkl'
# Feature Names
FEATURES = ["Active Power", "Guide Vane", "Pressure Down", "Pressure Up"]


In [13]:

def run_visualization():
    # 1. Load the data
    try:
        with open(DATA_PATH, 'rb') as f:
            data = pickle.load(f)
        X_data = data['X']
        # descriptions were saved in the previous script, so we load them
        descriptions = data['descriptions'] 
        y_labels = data['y']
        print(f"Successfully loaded {len(X_data)} samples.")
    except FileNotFoundError:
        print(f"Error: Could not find {DATA_PATH}. Run the generation cell first!")
        return

    # 2. Plotting Function
    def plot_interactive(anomaly_type, sample_index):
        # Filter indices by type
        type_indices = [i for i, x in enumerate(descriptions) if x == anomaly_type]
        
        if not type_indices:
            print(f"No samples found for {anomaly_type}")
            return
            
        # Select specific sample
        # Use modulo (%) so the slider wraps around if you go past the end
        actual_idx = type_indices[sample_index % len(type_indices)]
        
        # Find the "Normal" version of this sample
        # Since we generated 8 types per 1 original sample, the block size is 8
        # (Normal, Leak, Delay, Offset, Drift, Hammer, Jerky, Dropout)
        num_types = 8 
        normal_idx = (actual_idx // num_types) * num_types
        
        # Get signals
        anom_sig = X_data[actual_idx]
        norm_sig = X_data[normal_idx]
        
        # Plot
        fig, axes = plt.subplots(2, 2, figsize=(15, 8))
        fig.suptitle(f"Type: {anomaly_type} | Index: {actual_idx}", fontsize=16, fontweight='bold')
        
        for i, ax in enumerate(axes.flatten()):
            # Plot Normal (Gray dashed)
            ax.plot(norm_sig[:, i], color='gray', linestyle='--', alpha=0.6, label='Normal Baseline')
            # Plot Anomaly (Red solid)
            ax.plot(anom_sig[:, i], color='red', linewidth=1.5, label='Anomaly')
            
            ax.set_title(FEATURES[i])
            ax.grid(True, alpha=0.3)
            if i == 0: ax.legend(loc='upper right')
        
        plt.tight_layout()
        plt.show()

    # 3. Create Widgets
    unique_types = sorted(list(set(descriptions)), key=descriptions.index)

    dropdown = widgets.Dropdown(
        options=unique_types,
        value='Water Hammer Spike', # Default start
        description='Fault Type:',
        style={'description_width': 'initial'}
    )

    slider = widgets.IntSlider(
        min=0, max=50, step=1, value=0,
        description='Sample ID:',
        layout=Layout(width='400px')
    )

    # 4. Launch
    interact(plot_interactive, anomaly_type=dropdown, sample_index=slider)


In [ ]:

# Run the function
run_visualization()

Successfully loaded 4500 samples.


interactive(children=(Dropdown(description='Fault Type:', index=5, options=('Normal', 'Seal Leakage', 'Delayed…